# Merge Counts of Vulnerabilities and Libraries

In [8]:
import pandas as pd, re
from glob import glob

def sanitize(s):
    return re.sub(r'[^\w\-]', '_', str(s).strip())

main = pd.read_csv("/Users/dmk6603/Documents/swdb_opensource/1-indentify_open_source/MAIN.csv")
libs = pd.read_csv("/Users/dmk6603/Documents/swdb_opensource/5-generate_sbom/libraries.csv")
vulns = pd.read_csv("/Users/dmk6603/Documents/swdb_opensource/5.1-grype_vulnerabilities/vulnerabilities.csv")

main["project"] = main.apply(lambda r: f"{sanitize(r['VendorName'])}-{sanitize(r['Product'])}", axis=1)

# --- source files ---
sbom_files = {os.path.basename(f).replace(".cdx.json", ""): os.path.basename(f)
              for f in glob("sboms/*.cdx.json")}
vuln_files = {re.sub(r'[._]vulns\.json$', '', os.path.basename(f)): os.path.basename(f)
              for f in glob("vulnerabilities/*.json")}
main["sbom_file"] = main["project"].map(sbom_files).fillna("")
main["vuln_file"] = main["project"].map(vuln_files).fillna("")

# --- library counts ---
lib_versions = libs.groupby("project").apply(lambda g: g[["name","version"]].drop_duplicates().shape[0]).reset_index(name="unique_lib_versions")

lib_counts = libs.groupby("project").agg(
    total_libraries=("name", "count"),
    unique_libraries=("name", "nunique"),
    syft_languages=("language", lambda x: "; ".join(sorted(x.dropna().unique()))),
    syft_pkg_types=("pkg_type", lambda x: "; ".join(sorted(x.dropna().unique()))),
).reset_index().merge(lib_versions, on="project")

# --- vuln counts + new grype stats ---
def safe_mean(s):
    s = pd.to_numeric(s, errors="coerce").dropna()
    return round(s.mean(), 2) if len(s) else ""

def safe_max(s):
    s = pd.to_numeric(s, errors="coerce").dropna()
    return round(s.max(), 2) if len(s) else ""

vuln_counts = vulns.groupby("project").agg(
    total_vulns=("vuln_id", "count"),
    unique_cves=("cve", lambda x: x[x != ""].nunique()),
    critical=("severity", lambda x: (x == "Critical").sum()),
    high=("severity", lambda x: (x == "High").sum()),
    medium=("severity", lambda x: (x == "Medium").sum()),
    low=("severity", lambda x: (x == "Low").sum()),
    fixable=("fix_state", lambda x: (x == "fixed").sum()),
    not_fixed=("fix_state", lambda x: (x == "not-fixed").sum()),
    avg_cvss=("cvss_score", safe_mean),
    max_cvss=("cvss_score", safe_max),
    avg_risk=("risk_score", safe_mean),
    max_risk=("risk_score", safe_max),
    avg_epss=("epss", safe_mean),
    max_epss=("epss", safe_max),
    grype_languages=("pkg_language", lambda x: "; ".join(sorted(x.dropna().unique()))),
    grype_pkg_types=("pkg_type", lambda x: "; ".join(sorted(x.dropna().unique()))),
    cwes=("cwe", lambda x: "; ".join(sorted(set(c for c in x.dropna() if c != "")))),
).reset_index()

# pct fixable
vuln_counts["pct_fixable"] = (vuln_counts["fixable"] / vuln_counts["total_vulns"] * 100).round(2)

# vulnerable lib count + pct
vuln_libs = vulns.groupby("project")["pkg_name"].nunique().reset_index()
vuln_libs.columns = ["project", "vuln_lib_count"]
lib_counts = lib_counts.merge(vuln_libs, on="project", how="left")
lib_counts["vuln_lib_count"] = lib_counts["vuln_lib_count"].fillna(0)
lib_counts["pct_libs_vulnerable"] = (lib_counts["vuln_lib_count"] / lib_counts["unique_libraries"] * 100).round(2)

# --- merge ---
main = main.merge(lib_counts, on="project", how="left")
main = main.merge(vuln_counts, on="project", how="left")

# % of total enterprises
total_ent_col = "Total Enterprises"
main[total_ent_col] = main[total_ent_col].astype(str).str.replace(",", "").astype(float)
total_enterprises = main[total_ent_col].sum()
main["pct_enterprises"] = (main[total_ent_col] / total_enterprises * 100).round(2)

# active
main["gh_pushed_at"] = pd.to_datetime(main["gh_pushed_at"], errors="coerce")
cutoff = pd.Timestamp.now(tz="UTC") - pd.Timedelta(days=30)
main["active"] = (
    (main["gh_pushed_at"] >= cutoff) &
    (main["gh_stars"].fillna(0) >= 40) &
    (main["gh_forks"].fillna(0) >= 40)
)

main.drop(columns=["project"], inplace=True)
main.to_csv("MAIN.csv", index=False)
print(f"Done. Merged {len(lib_counts)} lib counts + {len(vuln_counts)} vuln counts into MAIN.csv")

/var/folders/fj/wtzx880x4v7g0q54zrwpf4gc0000gr/T/ipykernel_80858/3257300564.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  lib_versions = libs.groupby("project").apply(lambda g: g[["name","version"]].drop_duplicates().shape[0]).reset_index(name="unique_lib_versions")


Done. Merged 269 lib counts + 163 vuln counts into MAIN.csv
